# Customers — Spark SQL Analytics
Query customers and related data from HDFS using Spark SQL.

**Tables loaded:** `customers`, `orders`, `order_items`, `products`, `locations`

> Make sure CSVs are uploaded to HDFS at `hdfs:///user/hadoop/assessment-2/`


In [ ]:
import os, sys

# 1. Set env vars FIRST — before any pyspark import
os.environ["JAVA_HOME"]             = "/usr/local/java"
os.environ["SPARK_HOME"]            = "/usr/local/spark"
os.environ["HADOOP_CONF_DIR"]       = "/usr/local/hadoop/etc/hadoop"
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 2. Init findspark so pyspark JARs are on the path
import findspark
findspark.init()

# 3. NOW import pyspark (after findspark has set up the correct paths)
from pyspark.sql import SparkSession

spark = SparkSession.getActiveSession()
if spark is None:
    spark = (SparkSession.builder
        .appName("Customers SparkSQL")
        .master("local[*]")
        .config("spark.sql.shuffle.partitions", "4")
        .getOrCreate())

spark.sparkContext.setLogLevel("WARN")

# CSVs were uploaded to /data via uploadfile.sh
HDFS_BASE = "hdfs:///data"

def load_csv(name, renames):
    df = spark.read.option("header","true").option("inferSchema","true").csv(f"{HDFS_BASE}/{name}.csv")
    for old, new in renames.items():
        df = df.withColumnRenamed(old, new)
    df.createOrReplaceTempView(name)
    return df

customers   = load_csv("customers",   {"Customer ID":"customer_id","Customer Name":"customer_name","Segment":"segment"})
orders      = load_csv("orders",      {"Order ID":"order_id","Order Date":"order_date","Ship Date":"ship_date","Ship Mode":"ship_mode","Customer ID":"customer_id","Postal Code":"postal_code"})
order_items = load_csv("order_items", {"Row ID":"row_id","Order ID":"order_id","Product ID":"product_id","Sales":"sales","Quantity":"quantity","Discount":"discount","Profit":"profit"})
products    = load_csv("products",    {"Product ID":"product_id","Product Name":"product_name","Category":"category","Sub-Category":"sub_category"})
locations   = load_csv("locations",   {"Postal Code":"postal_code","City":"city","State":"state","Country":"country","Region":"region"})

print("All tables loaded:")
for t in ["customers","orders","order_items","products","locations"]:
    print(f"  {t}: {spark.table(t).count()} rows")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/27 17:17:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


TypeError: 'JavaPackage' object is not callable

In [ ]:
# ── Query 1: Full customer list ──────────────────────────────────────────────
print("=" * 60)
print("Query 1: Full Customer List")
print("=" * 60)
spark.sql("""
    SELECT
        customer_id,
        customer_name,
        segment
    FROM customers
    ORDER BY customer_name
""").show(100, truncate=False)

# ── Query 2: Customer count by segment ───────────────────────────────────────
print("=" * 60)
print("Query 2: Customer Count by Segment")
print("=" * 60)
spark.sql("""
    SELECT
        segment,
        COUNT(*) AS customer_count
    FROM customers
    GROUP BY segment
    ORDER BY customer_count DESC
""").show(truncate=False)

# ── Query 3: Top customers by total sales ────────────────────────────────────
print("=" * 60)
print("Query 3: Top 20 Customers by Total Sales")
print("=" * 60)
spark.sql("""
    SELECT
        c.customer_id,
        c.customer_name,
        c.segment,
        ROUND(SUM(oi.sales), 2)  AS total_sales,
        ROUND(SUM(oi.profit), 2) AS total_profit,
        COUNT(DISTINCT o.order_id) AS total_orders
    FROM customers c
    JOIN orders     o  ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id   = oi.order_id
    GROUP BY c.customer_id, c.customer_name, c.segment
    ORDER BY total_sales DESC
    LIMIT 20
""").show(truncate=False)

# ── Query 4: Customers with no orders ────────────────────────────────────────
print("=" * 60)
print("Query 4: Customers with No Orders")
print("=" * 60)
spark.sql("""
    SELECT
        c.customer_id,
        c.customer_name,
        c.segment
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.order_id IS NULL
""").show(truncate=False)

# ── Query 5: Customer distribution by region ─────────────────────────────────
print("=" * 60)
print("Query 5: Unique Customers per Region")
print("=" * 60)
spark.sql("""
    SELECT
        l.region,
        COUNT(DISTINCT c.customer_id) AS unique_customers,
        ROUND(SUM(oi.sales), 2)        AS total_sales
    FROM customers c
    JOIN orders      o  ON c.customer_id = o.customer_id
    JOIN locations   l  ON o.postal_code = l.postal_code
    JOIN order_items oi ON o.order_id    = oi.order_id
    GROUP BY l.region
    ORDER BY unique_customers DESC
""").show(truncate=False)

# ── Query 6: Average order value per customer segment ───────────────────────
print("=" * 60)
print("Query 6: Avg Order Value per Segment")
print("=" * 60)
spark.sql("""
    SELECT
        c.segment,
        ROUND(AVG(order_total), 2) AS avg_order_value,
        ROUND(SUM(order_total), 2) AS segment_total_sales
    FROM customers c
    JOIN (
        SELECT o.customer_id, o.order_id, SUM(oi.sales) AS order_total
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        GROUP BY o.customer_id, o.order_id
    ) ord ON c.customer_id = ord.customer_id
    GROUP BY c.segment
    ORDER BY avg_order_value DESC
""").show(truncate=False)
